# Pixels to Predictions — v2 Pipeline (Target: 0.90+)

## Recipe
- **Base**: 0.859 team's proven config (r=8, all 7 targets, flat LR=2e-4, 3 epochs, train+val combined)
- **Add**: Self-captioning, DoRA, subject/grade metadata, fixed inference, 5-model ensemble

## Score targets
- 0.859 baseline (proven) → +caption → +DoRA → +metadata → +ensemble → **0.90+**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Mount Drive & Setup
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_v2"
SUBMISSION_DIR = PROJECT_ROOT / "submissions_v2"
CAPTION_DIR = PROJECT_ROOT / "captions"

for d in [CHECKPOINT_DIR, SUBMISSION_DIR, CAPTION_DIR]:
    d.mkdir(exist_ok=True)

print("Data:", sorted(os.listdir(DATA_DIR)))
print("Checkpoints v2:", sorted(os.listdir(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else "empty")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Install
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Imports
# ══════════════════════════════════════════════════════════════════════════════
import json, random, time, gc, copy
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
MAX_CONTEXT_CHARS = 400  # per field, matches 0.859 team

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f} GB")
    USE_BF16 = torch.cuda.is_bf16_supported()
    COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    print(f"Compute dtype: {COMPUTE_DTYPE}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: Load Data (train+val COMBINED for training)
# ══════════════════════════════════════════════════════════════════════════════
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# CRITICAL: Combine train + val for training (0.859 team's biggest gain)
combined_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Combined: {len(combined_df)} | Test: {len(test_df)}")
print(f"Solutions available: {combined_df['solution'].notna().sum()}/{len(combined_df)}")
print(f"\n⚠️ Training on train+val combined. No validation — blind submit.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: Load Cached Captions
# ══════════════════════════════════════════════════════════════════════════════
caption_file = CAPTION_DIR / "all_captions.json"

if caption_file.exists():
    with open(caption_file) as f:
        all_captions = json.load(f)
    print(f"Loaded {len(all_captions)} cached captions.")
else:
    print("ERROR: No cached captions found!")
    print("Run the captioning stage from the v1 notebook first.")
    print(f"Expected at: {caption_file}")
    all_captions = {}  # will proceed without captions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Prompt Builders
# ══════════════════════════════════════════════════════════════════════════════

def _trunc(text, max_chars=MAX_CONTEXT_CHARS):
    if pd.isna(text): return ""
    text = str(text).strip()
    if len(text) <= max_chars: return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def prompt_caption_meta(row, captions, include_answer=False):
    """
    Starter format + image caption + subject/grade metadata.
    Matches 0.859 team's base prompt but with our additions.
    """
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    lecture = _trunc(row.get("lecture", ""))
    hint = _trunc(row.get("hint", ""))
    caption = captions.get(str(row["id"]), "")
    subject = str(row.get("subject", "")).strip() if pd.notna(row.get("subject")) else ""
    grade = str(row.get("grade", "")).strip() if pd.notna(row.get("grade")) else ""

    context_parts = []
    if caption:
        context_parts.append(f"Image description: {caption[:300]}")
    if lecture:
        context_parts.append(lecture)
    if hint:
        context_parts.append(hint)
    context_str = "\n".join(context_parts)

    prompt = "<image>\n"
    if subject or grade:
        meta = []
        if subject: meta.append(f"Subject: {subject}")
        if grade: meta.append(f"Grade: {grade}")
        prompt += " | ".join(meta) + "\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}"

    return prompt


def prompt_caption_meta_solution(row, captions, include_answer=False):
    """
    Same as above but training answer includes solution explanation.
    """
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    lecture = _trunc(row.get("lecture", ""))
    hint = _trunc(row.get("hint", ""))
    caption = captions.get(str(row["id"]), "")
    subject = str(row.get("subject", "")).strip() if pd.notna(row.get("subject")) else ""
    grade = str(row.get("grade", "")).strip() if pd.notna(row.get("grade")) else ""

    context_parts = []
    if caption:
        context_parts.append(f"Image description: {caption[:300]}")
    if lecture:
        context_parts.append(lecture)
    if hint:
        context_parts.append(hint)
    context_str = "\n".join(context_parts)

    prompt = "<image>\n"
    if subject or grade:
        meta = []
        if subject: meta.append(f"Subject: {subject}")
        if grade: meta.append(f"Grade: {grade}")
        prompt += " | ".join(meta) + "\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}"
        solution = _trunc(row.get("solution", ""), 200)
        if solution:
            prompt += f" Explanation: {solution}"

    return prompt


def prompt_no_caption(row, captions, include_answer=False):
    """
    Exact 0.859 team prompt — no caption, no metadata. For diversity.
    """
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    lecture = _trunc(row.get("lecture", ""))
    hint = _trunc(row.get("hint", ""))

    context_parts = []
    if lecture: context_parts.append(lecture)
    if hint: context_parts.append(hint)
    context_str = "\n".join(context_parts)

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}"

    return prompt


# Test
row = combined_df.iloc[0]
print("=== Caption + Metadata ===")
print(prompt_caption_meta(row, all_captions, True)[:500])
print("\n=== No Caption (0.859 team format) ===")
print(prompt_no_caption(row, all_captions, True)[:500])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Dataset + Collator (answer-letter-only loss, matching 0.859 team)
# ══════════════════════════════════════════════════════════════════════════════

class TrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, captions, prompt_fn):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.captions = captions
        self.prompt_fn = prompt_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.data_dir / row["image_path"]).convert("RGB")

        full_text = self.prompt_fn(row, self.captions, include_answer=True)
        prompt_text = self.prompt_fn(row, self.captions, include_answer=False)

        full_enc = self.processor(
            text=[full_text], images=[image],
            return_tensors="pt", truncation=False
        )
        prompt_enc = self.processor(
            text=[prompt_text], images=[image],
            return_tensors="pt", truncation=False
        )

        return {
            "input_ids": full_enc["input_ids"].squeeze(0),
            "attention_mask": full_enc["attention_mask"].squeeze(0),
            "pixel_values": full_enc["pixel_values"].squeeze(0),
            "prompt_len": prompt_enc["input_ids"].shape[1],
        }


def collate_fn(batch, pad_id):
    """Left-pad, loss on answer tokens only."""
    max_len = max(x["input_ids"].shape[0] for x in batch)
    ids, masks, labels, pixels = [], [], [], []

    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len

        padded_ids = torch.cat([torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype), x["input_ids"]])
        padded_mask = torch.cat([torch.zeros(pad_len, dtype=x["attention_mask"].dtype), x["attention_mask"]])

        lab = torch.full_like(padded_ids, -100)
        answer_start = pad_len + x["prompt_len"]
        lab[answer_start:] = padded_ids[answer_start:]

        ids.append(padded_ids)
        masks.append(padded_mask)
        labels.append(lab)
        pixels.append(x["pixel_values"])

    return {
        "input_ids": torch.stack(ids),
        "attention_mask": torch.stack(masks),
        "labels": torch.stack(labels),
        "pixel_values": torch.stack(pixels),
    }

print("Dataset + Collator ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Model Loader (QLoRA + DoRA support)
# ══════════════════════════════════════════════════════════════════════════════
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

def load_model(seed, lora_r=8, lora_alpha=16, use_dora=False,
               targets=None):
    """Load base model with QLoRA (optionally DoRA)."""
    if targets is None:
        targets = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]  # all 7

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=targets,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        use_dora=use_dora,
    )
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    dora_str = "DoRA" if use_dora else "LoRA"
    print(f"{dora_str} r={lora_r} alpha={lora_alpha} | Trainable: {trainable:,} ({trainable/1e6:.2f}M) | "
          f"{'OK' if trainable <= 5_000_000 else 'OVER 5M LIMIT!'}")

    return model, processor

print("Model loader ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Training Function (flat LR, 3 epochs, matching 0.859 team)
# ══════════════════════════════════════════════════════════════════════════════

def train_model(model, processor, train_df, captions, prompt_fn,
                exp_name, seed=42, epochs=3, lr=2e-4,
                batch_size=4, grad_accum=2):
    """
    Train with FLAT LR (no scheduler), matching the 0.859 team's setup.
    Checkpoints every 25% of each epoch.
    """
    print(f"\n{'='*60}")
    print(f"TRAINING: {exp_name}")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pad_id = processor.tokenizer.pad_token_id
    dataset = TrainDataset(train_df, processor, DATA_DIR, captions, prompt_fn)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_id),
        num_workers=2, pin_memory=True,
    )

    # FLAT LR — no scheduler (0.859 team finding: cosine kills epoch 3)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.0,
    )

    total_batches = len(loader) * epochs
    print(f"Batches/epoch: {len(loader)} | Total batches: {total_batches}")
    print(f"LR: {lr} (FLAT) | Grad accum: {grad_accum} | Eff batch: {batch_size * grad_accum}")

    model.train()
    log = []
    start_time = time.time()

    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()

        pbar = tqdm(enumerate(loader), total=len(loader),
                    desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch_idx, batch in pbar:
            batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss / grad_accum
            loss.backward()

            epoch_losses.append(outputs.loss.item())

            if (batch_idx + 1) % grad_accum == 0 or (batch_idx + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            avg_loss = np.mean(epoch_losses[-50:])
            pbar.set_postfix({"loss": f"{outputs.loss.item():.4f}", "avg": f"{avg_loss:.4f}",
                              "VRAM": f"{torch.cuda.memory_allocated()/1e9:.1f}GB"})

            if (batch_idx + 1) % 100 == 0:
                log.append({"epoch": epoch+1, "batch": batch_idx+1,
                            "loss": outputs.loss.item(), "avg": avg_loss})
                print(f"  Step {batch_idx+1}/{len(loader)} | loss={outputs.loss.item():.4f} avg={avg_loss:.4f}")

            # Checkpoint every 25%
            ckpt_interval = max(len(loader) // 4, 1)
            if (batch_idx + 1) % ckpt_interval == 0 and (batch_idx + 1) < len(loader):
                p = CHECKPOINT_DIR / f"{exp_name}_ep{epoch+1}_b{batch_idx+1}"
                model.save_pretrained(str(p))
                print(f"  Checkpoint: {p.name}")

        epoch_avg = np.mean(epoch_losses)
        print(f"\nEpoch {epoch+1} — Avg loss: {epoch_avg:.4f}")

        # Save end-of-epoch checkpoint
        ep_path = CHECKPOINT_DIR / f"{exp_name}_ep{epoch+1}_final"
        model.save_pretrained(str(ep_path))
        print(f"  Epoch checkpoint: {ep_path.name}")

    elapsed = time.time() - start_time
    final_path = CHECKPOINT_DIR / f"{exp_name}_final"
    model.save_pretrained(str(final_path))

    with open(CHECKPOINT_DIR / f"{exp_name}_log.json", "w") as f:
        json.dump(log, f, indent=2)

    print(f"\nModel saved: {final_path.name} ({elapsed/60:.1f} min)")

    del optimizer, loader, dataset
    gc.collect()
    torch.cuda.empty_cache()

    return model

print("Training function ready (flat LR, 3 epochs).")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: Inference (fixed padding + multi-form tokens)
# ══════════════════════════════════════════════════════════════════════════════

def get_candidate_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        ids = set()
        for form in [letter, " " + letter]:
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids


def predict_batch(model, processor, df_batch, captions, prompt_fn, cand_ids):
    """Score batch with FIXED padding position."""
    images = [Image.open(DATA_DIR / row["image_path"]).convert("RGB")
              for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row, captions, include_answer=False)
               for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits

    seq_lengths = inputs["attention_mask"].sum(dim=1)
    preds, scores_all = [], []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        last_pos = int(seq_lengths[i].item()) - 1
        lp = torch.log_softmax(logits[i, last_pos, :], dim=-1)
        scores = [max(lp[tid].item() for tid in cand_ids[CHOICE_LETTERS[ci]])
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all


def run_inference(model, processor, df, captions, prompt_fn, batch_size=4, desc="Infer"):
    """Full inference with progress."""
    model.eval()
    cand_ids = get_candidate_ids(processor.tokenizer)
    all_preds, all_scores = [], []

    pbar = tqdm(range(0, len(df), batch_size), desc=desc, leave=True)
    for start in pbar:
        end = min(start + batch_size, len(df))
        preds, scores = predict_batch(model, processor, df.iloc[start:end],
                                       captions, prompt_fn, cand_ids)
        all_preds.extend(preds)
        all_scores.extend(scores)
        pbar.set_postfix({"done": len(all_preds)})
        torch.cuda.empty_cache()

    return all_preds, all_scores

print("Inference ready (fixed padding).")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: Ensemble Configs
# ══════════════════════════════════════════════════════════════════════════════

CONFIGS = [
    {
        "name": "v2_cap_meta_dora_r8_s42",
        "seed": 42, "lora_r": 8, "lora_alpha": 16,
        "use_dora": True,
        "prompt_fn": prompt_caption_meta,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "v2_cap_meta_dora_r8_s123",
        "seed": 123, "lora_r": 8, "lora_alpha": 16,
        "use_dora": True,
        "prompt_fn": prompt_caption_meta,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "v2_cap_sol_dora_r8_s456",
        "seed": 456, "lora_r": 8, "lora_alpha": 16,
        "use_dora": True,
        "prompt_fn": prompt_caption_meta_solution,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "v2_cap_meta_dora_a32_r8_s789",
        "seed": 789, "lora_r": 8, "lora_alpha": 32,
        "use_dora": True,
        "prompt_fn": prompt_caption_meta,
        "lr": 2e-4, "epochs": 3,
    },
    {
        "name": "v2_nocap_lora_r8_s42",
        "seed": 42, "lora_r": 8, "lora_alpha": 16,
        "use_dora": False,
        "prompt_fn": prompt_no_caption,
        "lr": 2e-4, "epochs": 3,
    },
]

print(f"{len(CONFIGS)} ensemble configs:")
for c in CONFIGS:
    dora = "DoRA" if c["use_dora"] else "LoRA"
    print(f"  {c['name']}: {dora} r={c['lora_r']} a={c['lora_alpha']}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12: MAIN LOOP — Train + Infer ALL Models
# ══════════════════════════════════════════════════════════════════════════════

all_results = []

for i, cfg in enumerate(CONFIGS):
    print(f"\n\n{'#'*60}")
    print(f"# MODEL {i+1}/{len(CONFIGS)}: {cfg['name']}")
    print(f"{'#'*60}")

    results_path = CHECKPOINT_DIR / f"{cfg['name']}_results.json"

    # CACHE CHECK: skip if already done
    if results_path.exists():
        print(f"Loading cached results: {results_path.name}")
        with open(results_path) as f:
            cached = json.load(f)
        all_results.append(cached)
        continue

    # Load fresh model
    model, processor = load_model(
        seed=cfg["seed"], lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"], use_dora=cfg["use_dora"],
    )

    # Train on COMBINED train+val
    model = train_model(
        model, processor, combined_df, all_captions,
        prompt_fn=cfg["prompt_fn"],
        exp_name=cfg["name"],
        seed=cfg["seed"],
        epochs=cfg["epochs"],
        lr=cfg["lr"],
        batch_size=4,  # matches 0.859 team
        grad_accum=2,  # effective batch = 8
    )

    # Test inference
    print(f"\nRunning test inference...")
    test_preds, test_scores = run_inference(
        model, processor, test_df, all_captions,
        prompt_fn=cfg["prompt_fn"],
        batch_size=4, desc=f"Test ({cfg['name'][:25]})"
    )

    # Save results
    result = {
        "name": cfg["name"],
        "test_preds": test_preds,
        "test_scores": test_scores,
        "config": {k: str(v) if callable(v) else v
                   for k, v in cfg.items() if k != "prompt_fn"},
    }
    all_results.append(result)

    with open(results_path, "w") as f:
        json.dump(result, f)

    # Save individual submission
    sub = pd.DataFrame({"id": test_df["id"], "answer": test_preds})
    sub.to_csv(SUBMISSION_DIR / f"sub_{cfg['name']}.csv", index=False)
    print(f"Saved: sub_{cfg['name']}.csv")

    # Free model
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM freed: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print(f"\n{'='*60}")
print(f"ALL {len(CONFIGS)} MODELS COMPLETE")
print(f"{'='*60}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13: Ensemble + Final Submissions
# ══════════════════════════════════════════════════════════════════════════════

test_num_choices = [len(row["choices"]) for _, row in test_df.iterrows()]

# Majority vote
def majority_vote(pred_lists):
    return [Counter([pl[i] for pl in pred_lists]).most_common(1)[0][0]
            for i in range(len(pred_lists[0]))]

# Score average
def score_avg(score_lists, nc_list):
    preds = []
    for i in range(len(score_lists[0])):
        avg = [np.mean([sl[i][ci] for sl in score_lists if ci < len(sl[i])])
               for ci in range(nc_list[i])]
        preds.append(int(np.argmax(avg)))
    return preds

pred_lists = [r["test_preds"] for r in all_results]
score_lists = [r["test_scores"] for r in all_results]

mv_preds = majority_vote(pred_lists)
sa_preds = score_avg(score_lists, test_num_choices)

# Save all submissions
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

submissions = {
    "majority_vote": mv_preds,
    "score_average": sa_preds,
}

# Also add each individual model
for r in all_results:
    submissions[r["name"]] = r["test_preds"]

for name, preds in submissions.items():
    sub = pd.DataFrame({"id": test_df["id"], "answer": preds})
    assert len(sub) == len(sample_sub)
    sub.to_csv(SUBMISSION_DIR / f"sub_{name}.csv", index=False)

# Default submission = score_average (preserves confidence)
best_sub = pd.DataFrame({"id": test_df["id"], "answer": sa_preds})
best_sub.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

print(f"{'='*60}")
print(f"SUBMISSIONS SAVED")
print(f"{'='*60}")
for name in submissions:
    print(f"  sub_{name}.csv")
print(f"\n  submission.csv (= score_average)")
print(f"\nPrediction distribution (score_average):")
print(best_sub["answer"].value_counts().sort_index())
print(f"\nAll files at: {SUBMISSION_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14: Save Summary + Kill Runtime
# ══════════════════════════════════════════════════════════════════════════════

summary = {
    "n_models": len(all_results),
    "models": [r["name"] for r in all_results],
    "training_data": "train+val combined (4,157 samples)",
    "submissions_generated": list(submissions.keys()),
    "techniques": [
        "0.859 team recipe: r=8, all 7 targets, flat LR=2e-4, 3 epochs, train+val",
        "Self-captioning (SmolVLM image descriptions)",
        "DoRA (magnitude+direction decomposition)",
        "Subject/grade metadata in prompt",
        "Solution field in training (1 model)",
        "Fixed padding position inference",
        "Multi-form token scoring",
        "5-model diverse ensemble (majority vote + score average)",
    ],
}

with open(CHECKPOINT_DIR / "summary_v2.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved.")
print(json.dumps(summary, indent=2))

# Auto-kill runtime to save credits
print("\nKilling runtime to save credits...")
import os
os.kill(os.getpid(), 9)